In [ ]:
import subprocess, sys
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'orjson', 'polars', 'pyarrow'])
import os, gc, gzip, orjson, random, csv
import pandas as pd
import numpy as np
import polars as pl
import pyarrow as pa
import pyarrow.parquet as pq

In [ ]:
# Cấu hình đường dẫn
RAW_REVIEW_PATH = '/kaggle/input/datasets/b22dckh072/file01/Clothing_Shoes_and_Jewelry.jsonl/Clothing_Shoes_and_Jewelry.jsonl'
RAW_META_PATH   = '/kaggle/input/datasets/b22dckh072/file01/meta_Clothing_Shoes_and_Jewelry.jsonl/meta_Clothing_Shoes_and_Jewelry.jsonl'

WORKING_DIR = '/kaggle/working/processed'
os.makedirs(WORKING_DIR, exist_ok=True)

INTERIM_RAW_PATH = '/kaggle/working/interim_raw.parquet'
TRAIN_OUT_PATH = os.path.join(WORKING_DIR, 'train_interactions.parquet')
TEST_OUT_PATH  = os.path.join(WORKING_DIR, 'test_interactions.parquet')
META_OUT_PATH  = os.path.join(WORKING_DIR, 'filtered_metadata.parquet')

K_CORE = 5
TRAIN_RATIO = 0.8
CHUNK_SIZE = 3000000 

In [ ]:
def jsonl_to_parquet(input_path, output_path, chunk_size):
    print("BƯỚC 1: Chuyển đổi file Review 27GB sang Parquet...")
    writer = None
    buffer = []
    chunk_count = 0

    with open(input_path, 'r', encoding='utf-8') as f:
        for line in f:
            if not line.strip(): continue
            try:
                data = orjson.loads(line)
                buffer.append({
                    'user_id': data.get('user_id'),
                    'parent_asin': data.get('parent_asin'),
                    'rating': data.get('rating'),
                    'timestamp': data.get('timestamp')
                })
                
                # Khi buffer đầy, xả xuống ổ cứng
                if len(buffer) >= chunk_size:
                    chunk_count += 1
                    table = pa.Table.from_pandas(pd.DataFrame(buffer))
                    if writer is None:
                        writer = pq.ParquetWriter(output_path, table.schema)
                    writer.write_table(table)
                    buffer.clear()
                    print(f"   Đã ghi xong Chunk thứ {chunk_count}")
                    gc.collect()
            except Exception:
                continue # Bỏ qua dòng lỗi do giải nén hỏng

    # Ghi phần dư
    if buffer:
        table = pa.Table.from_pandas(pd.DataFrame(buffer))
        if writer is None:
            writer = pq.ParquetWriter(output_path, table.schema)
        writer.write_table(table)

    if writer: writer.close()
    print("-> Hoàn tất nạp dữ liệu thô!")

In [ ]:
def apply_k_core(data_path, k=5):
    print(f"BƯỚC 2: Áp dụng K-Core={k}...")
    lf = pl.scan_parquet(data_path).select(['user_id', 'parent_asin'])
    curr_data = lf.collect()

    for i in range(3):
        print(f"   Vòng lặp K-Core thứ {i+1}...")
        user_counts = curr_data.group_by('user_id').len()
        valid_users = user_counts.filter(pl.col('len') >= k).select('user_id')
        curr_data = curr_data.join(valid_users, on='user_id', how='inner')

        item_counts = curr_data.group_by('parent_asin').len()
        valid_items = item_counts.filter(pl.col('len') >= k).select('parent_asin')
        curr_data = curr_data.join(valid_items, on='parent_asin', how='inner')

        del user_counts, valid_users, item_counts, valid_items
        gc.collect()

    return curr_data.unique()

In [ ]:
def split_and_map(raw_path, valid_ids, train_out, test_out, ratio):
    print("BƯỚC 3: Map ID và Phân tách Train/Test theo MỐC THỜI GIAN CHUNG...")

    valid_user_set = set(valid_ids['user_id'].to_list())
    valid_item_set = set(valid_ids['parent_asin'].to_list())

    u_map = pl.DataFrame({'user_id': list(valid_user_set)}).with_row_index('mapped_user_id', offset=1)
    i_map = pl.DataFrame({'parent_asin': list(valid_item_set)}).with_row_index('mapped_item_id', offset=1)
    
    u_map = u_map.with_columns(pl.col('mapped_user_id').cast(pl.Int32))
    i_map = i_map.with_columns(pl.col('mapped_item_id').cast(pl.Int32))

    # Nạp dữ liệu và gắn ID
    df_clean = (pl.scan_parquet(raw_path)
                .join(u_map.lazy(), on='user_id', how='inner')
                .join(i_map.lazy(), on='parent_asin', how='inner')
                .select(['mapped_user_id', 'mapped_item_id', 'rating', 'timestamp'])
                .collect())

    df_clean = df_clean.sort('timestamp')
    
    cutoff_idx = int(df_clean.height * ratio)
    
    # Rút ra mốc thời gian (timestamp) tại dòng đó làm chuẩn chung
    cutoff_timestamp = df_clean['timestamp'][cutoff_idx]
    print(f"-> Mốc thời gian chung để cắt hệ thống là: {cutoff_timestamp}")

    train_data = df_clean.filter(pl.col('timestamp') <= cutoff_timestamp)
    test_data  = df_clean.filter(pl.col('timestamp') > cutoff_timestamp)

    print("-> Đang loại bỏ các User/Item mới xuất hiện trong Test mà Train chưa có...")
    train_users = train_data.select('mapped_user_id').unique()
    train_items = train_data.select('mapped_item_id').unique()
    
    test_data = (test_data
                 .join(train_users, on='mapped_user_id', how='inner')
                 .join(train_items, on='mapped_item_id', how='inner'))

    # Ghi xuống đĩa cứng
    train_data.write_parquet(train_out)
    test_data.write_parquet(test_out)

    print(f"-> Train: {train_data.height:,} dòng | Test: {test_data.height:,} dòng.")
    return valid_item_set, i_map

In [ ]:
def process_meta(meta_path, valid_items_set, item_map_df, output_path, chunk_size):
    import re
    print("BƯỚC 4: Lọc và Xử lý Meta Data 17GB...")
    writer = None
    buffer = []
    
    # Hàm con: Làm sạch và ép kiểu về số thập phân
    def clean_number(val):
        try:
            if isinstance(val, str):
                # Rút trích phần số thực khỏi chuỗi (VD: "$10.99" hoặc "—" -> 10.99 hoặc 0.0)
                nums = re.findall(r'\d+\.?\d*', val)
                return float(nums[0]) if nums else 0.0
            return float(val) if val is not None else 0.0
        except:
            return 0.0

    with open(meta_path, 'r', encoding='utf-8') as f:
        for line in f:
            if not line.strip(): continue
            try:
                data = orjson.loads(line)
                asin = data.get('parent_asin')

                if asin in valid_items_set:
                    buffer.append({
                        'parent_asin': asin,
                        'price': clean_number(data.get('price')), # Đã bọc hàm làm sạch
                        'average_rating': clean_number(data.get('average_rating')),
                        'rating_number': int(clean_number(data.get('rating_number'))),
                        'store': str(data.get('store', 'Unknown'))
                    })

                if len(buffer) >= chunk_size:
                    df_chunk = pd.DataFrame(buffer)
                    table = pa.Table.from_pandas(df_chunk)
                    if writer is None:
                        writer = pq.ParquetWriter(output_path + ".tmp", table.schema)
                    writer.write_table(table)
                    buffer.clear()
                    gc.collect()

            except Exception:
                continue

    if buffer:
        df_chunk = pd.DataFrame(buffer)
        table = pa.Table.from_pandas(df_chunk)
        if writer is None:
            writer = pq.ParquetWriter(output_path + ".tmp", table.schema)
        writer.write_table(table)

    if writer: writer.close()

    print("   Đang gắn Map ID cho Meta Data...")
    (pl.scan_parquet(output_path + ".tmp")
     .join(item_map_df.lazy(), on='parent_asin', how='inner')
     .drop('parent_asin') 
     .sink_parquet(output_path))

    os.remove(output_path + ".tmp")
    print("-> Hoàn tất xử lý Meta Data!")

In [ ]:
jsonl_to_parquet(RAW_REVIEW_PATH, INTERIM_RAW_PATH, CHUNK_SIZE)
valid_ids_df = apply_k_core(INTERIM_RAW_PATH, K_CORE)
valid_items, item_mapping = split_and_map(INTERIM_RAW_PATH, valid_ids_df, TRAIN_OUT_PATH, TEST_OUT_PATH, TRAIN_RATIO)

MAPPING_OUT_PATH = os.path.join(WORKING_DIR, 'item_mapping.parquet')
item_mapping.write_parquet(MAPPING_OUT_PATH)
print(f"-> Đã lưu file Mapping tại: {MAPPING_OUT_PATH}")

process_meta(RAW_META_PATH, valid_items, item_mapping, META_OUT_PATH, CHUNK_SIZE)

# Dọn dẹp rác tạm thời
if os.path.exists(INTERIM_RAW_PATH):
    os.remove(INTERIM_RAW_PATH)
    
print("HOÀN TẤT TOÀN BỘ FILE 01!")